In [ ]:
# Simple demo: dynamics + observer + PD controller, with plotting

import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

# ============================
# Reuse the Spacecraft class
# ============================
class Spacecraft:
    def __init__(self, J, delta, C, K, theta=5.0, theta1=2.0, theta2=5.0):
        self.J = J
        self.delta = delta
        self.C = C
        self.K = K
        self.Jbar = J - delta.T @ delta
        self.N_modes = delta.shape[0]
        self.theta = theta
        self.theta1 = theta1
        self.theta2 = theta2

    def dynamics(self, t, X, Tr):
        q = X[0:3]
        v = X[3:6]
        eta = X[6:6 + self.N_modes]
        psi = X[6 + self.N_modes:6 + 2 * self.N_modes]

        T = self.T_matrix(q)
        P = np.linalg.inv(T)

        q_dot = v
        f = self.f_nonlinear(q, v, eta, psi)
        g = self.g_matrix(q)
        v_dot = f + g @ Tr

        eta_dot = psi - self.delta @ (P @ v)
        psi_dot = - self.C @ psi - self.K @ eta + self.C @ self.delta @ (P @ v)

        return np.concatenate([q_dot, v_dot, eta_dot, psi_dot])

    def observer(self, t, X_obs, Tr):
        q_hat = X_obs[0:3]
        v_hat = X_obs[3:6]
        eta_hat = X_obs[6:6 + self.N_modes]
        psi_hat = X_obs[6 + self.N_modes:6 + 2 * self.N_modes]

        q = self.q_true(t)
        q_tilde = q - q_hat

        T_hat = self.T_matrix(q_hat)
        P_hat = np.linalg.inv(T_hat)

        f_hat = self.f_nonlinear(q, v_hat, eta_hat, psi_hat)
        g = self.g_matrix(q)

        q_hat_dot = v_hat + self.theta * self.theta1 * q_tilde
        v_hat_dot = f_hat + g @ Tr + self.theta**2 * self.theta2 * q_tilde

        eta_hat_dot = psi_hat - self.delta @ (P_hat @ v_hat)
        psi_hat_dot = - self.C @ psi_hat - self.K @ eta_hat + self.C @ self.delta @ (P_hat @ v_hat)

        return np.concatenate([q_hat_dot, v_hat_dot, eta_hat_dot, psi_hat_dot])

    def f_nonlinear(self, q, v, eta, psi):
        T = self.T_matrix(q)
        P = np.linalg.inv(T)
        T_dot = self.T_dot(q, v)
        P_dot = -P @ T_dot @ P
        omega = P @ v
        omega_cross = self.skew(omega)

        term1 = - T @ (P_dot @ v)
        term2 = - T @ np.linalg.inv(self.Jbar) @ (
            omega_cross @ (self.Jbar @ omega + self.delta.T @ psi)
        )
        term3 = T @ np.linalg.inv(self.Jbar) @ (
            self.delta.T @ (self.C @ psi + self.K @ eta - self.C @ self.delta @ omega)
        )
        return term1 + term2 + term3

    def g_matrix(self, q):
        return self.T_matrix(q) @ np.linalg.inv(self.Jbar)

    def T_matrix(self, q):
        q_cross = self.skew(q)
        q_norm2 = q @ q
        return 0.5 * ((1 - q_norm2 / 2) * np.eye(3) + q_cross + np.outer(q, q))

    def T_dot(self, q, v):
        return 0.5 * (-(q @ v) * np.eye(3) + self.skew(v) + 2 * np.outer(q, v))

    def skew(self, x):
        return np.array([[0, -x[2], x[1]],
                         [x[2], 0, -x[0]],
                         [-x[1], x[0], 0]])

# ============================
# Parameters & initialization
# ============================
# J = np.diag([10.0, 8.0, 6.0])
# delta = np.array([[0.2, 0.1, 0.0]])
# C = np.diag([0.5])
# K = np.diag([2.0])


J = np.array([[120., 3., 4.],
            [3., 100., 10.],
            [4., 10., 120.]])
delta = np.array([
                    [ 6.45637,  1.27814,  2.15629],
                    [-1.25819,  0.91756, -1.67264],
                    [ 1.11687,  2.48901, -0.83674],
                    [ 1.23637, -2.65810, -1.12530]
                ])  # shape (4,3)
omega_n = np.array([0.7681, 1.1038, 1.8733, 2.5496])  # rad/s (4 modes)
xi = np.array([0.0056, 0.0086, 0.0128, 0.0252])
K = np.diag(omega_n**2)            # K = diag{omega_n^2}
C = np.diag(2.0 * xi * omega_n)    # C = diag{2*xi*omega_n}

sc = Spacecraft(J, delta, C, K)

# Initial conditions
# q0 = np.array([0.2, -0.1, 0.15])
# v0 = np.zeros(3)
# eta0 = np.array([0.1])
# psi0 = np.zeros(1)
q0 = np.array([0.3,0.1,-0.2])
v0 = np.zeros(3)
m = K.shape[0]
eta0 = np.zeros(m)
psi0 = np.zeros(m) 

Xv0 = np.concatenate([q0, v0, eta0, psi0])

# Observer initial guess (intentionally biased)
# Xobs0 = np.zeros_like(X0)
Xobsv0 = np.concatenate([q0, v0, eta0, psi0])

# Simple PD controller on attitude
Kp = 30
Kd = 70

def controller(q_hat, v_hat):
    return -Kp * q_hat - Kd * v_hat

# ============================
# Closed-loop simulation
# ============================
tspan = np.linspace(0, 100, 10001)

# Integrate true dynamics with observer-in-the-loop
Xv = np.zeros((len(Xv0), len(tspan)))
Xobsv = np.zeros_like(Xv)
Xv[:, 0] = Xv0
Xobsv[:, 0] = Xobsv0


#-------------- Added variables: with omega as state ---------------#
X    = np.zeros_like(Xv)
Xobs = np.zeros_like(Xv)

# Initial omega
P0 = np.linalg.inv(sc.T_matrix(q0))
omega0 = P0 @ v0

# Initialize new X / Xobs
X[:, 0]    = np.concatenate([q0, omega0, eta0, psi0])
Xobs[:, 0] = np.concatenate([q0*0, omega0*0, eta0*0, psi0*0])
#----------------------------------------------------#


for k in range(len(tspan) - 1):
    t0, t1 = tspan[k], tspan[k + 1]

    q_hat = Xobsv[0:3, k]
    v_hat = Xobsv[3:6, k]
    Tr = controller(q_hat, v_hat)

    sol_true = solve_ivp(
        lambda t, x: sc.dynamics(t, x, Tr),
        [t0, t1],
        Xv[:, k],
        method='LSODA'
    )

    Xv[:, k + 1] = sol_true.y[:, -1]

    # update q_true for observer
    sc.q_true = lambda t, q=Xv[0:3, k + 1]: q

    sol_obs = solve_ivp(
        lambda t, x: sc.observer(t, x, Tr),
        [t0, t1],
        Xobsv[:, k],
        method='LSODA'
    )

    Xobsv[:, k + 1] = sol_obs.y[:, -1]
    
    # =====================================================
    # Added section: maintain X / Xobs (omega form)
    # =====================================================

    # ---- True system ----
    q     = Xv[0:3, k + 1]
    v     = Xv[3:6, k + 1]
    eta   = Xv[6:6 + sc.N_modes, k + 1]
    psi   = Xv[6 + sc.N_modes:6 + 2 * sc.N_modes, k + 1]

    P = np.linalg.inv(sc.T_matrix(q))
    omega = P @ v

    X[:, k + 1] = np.concatenate([q, omega, eta, psi])

    # ---- Observer ----
    q_hat   = Xobsv[0:3, k + 1]
    v_hat   = Xobsv[3:6, k + 1]
    eta_hat = Xobsv[6:6 + sc.N_modes, k + 1]
    psi_hat = Xobsv[6 + sc.N_modes:6 + 2 * sc.N_modes, k + 1]

    P_hat = np.linalg.inv(sc.T_matrix(q_hat))
    omega_hat = P_hat @ v_hat

    Xobs[:, k + 1] = np.concatenate([q_hat, omega_hat, eta_hat, psi_hat])

# ============================
# Plot results
# ============================
# plt.figure()
# plt.plot(tspan, Xv[0, :])
# plt.plot(tspan, Xobsv[0, :])
# plt.xlabel("Time (s)")
# plt.ylabel("q1")
# plt.title("Attitude state vs observer estimate")
# plt.legend(["True", "Estimated"])
# plt.show()

# plt.figure()
# plt.plot(tspan, Xv[0, :] - Xobsv[0, :])
# plt.xlabel("Time (s)")
# plt.ylabel("Estimation error")
# plt.title("Observer estimation error (q1)")
# plt.show()
import matplotlib.pyplot as plt

plt.figure()
plt.plot(tspan, Xv[0, :])
plt.plot(tspan, Xobsv[0, :])
plt.plot(tspan, Xv[1, :])
plt.plot(tspan, Xobsv[1, :])
plt.plot(tspan, Xv[2, :])
plt.plot(tspan, Xobsv[2, :])

plt.xlabel("Time (s)")
plt.ylabel("MRP")
plt.title("Attitude state vs observer estimate (MRP)")
plt.legend([
    "q1 (true)", "q1 (estimated)",
    "q2 (true)", "q2 (estimated)",
    "q3 (true)", "q3 (estimated)"
])
plt.show()

plt.figure()
plt.plot(tspan, Xv[0, :] - Xobsv[0, :])
plt.plot(tspan, Xv[1, :] - Xobsv[1, :])
plt.plot(tspan, Xv[2, :] - Xobsv[2, :])

plt.xlabel("Time (s)")
plt.ylabel("MRP estimation error")
plt.title("Observer estimation error (MRP, v-based)")
plt.legend([
    "q1 error", "q2 error", "q3 error"
])
plt.show()

plt.figure()
plt.plot(tspan, X[3, :])
plt.plot(tspan, Xobs[3, :])
plt.plot(tspan, X[4, :])
plt.plot(tspan, Xobs[4, :])
plt.plot(tspan, X[5, :])
plt.plot(tspan, Xobs[5, :])

plt.xlabel("Time (s)")
plt.ylabel("Angular velocity (rad/s)")
plt.title("Angular velocity state vs observer estimate")
plt.legend([
    "ω1 (true)", "ω1 (estimated)",
    "ω2 (true)", "ω2 (estimated)",
    "ω3 (true)", "ω3 (estimated)"
])
plt.show()


plt.figure()
plt.plot(tspan, X[3, :] - Xobs[3, :])
plt.plot(tspan, X[4, :] - Xobs[4, :])
plt.plot(tspan, X[5, :] - Xobs[5, :])

plt.xlabel("Time (s)")
plt.ylabel("Angular velocity estimation error (rad/s)")
plt.title("Observer estimation error (angular velocity)")
plt.legend([
    "ω1 error", "ω2 error", "ω3 error"
])
plt.show()


plt.figure()
plt.plot(tspan, Xv[6, :])
plt.plot(tspan, Xobsv[6, :])
plt.plot(tspan, Xv[7, :])
plt.plot(tspan, Xobsv[7, :])
plt.plot(tspan, Xv[8, :])
plt.plot(tspan, Xobsv[8, :])
plt.plot(tspan, Xv[9, :])
plt.plot(tspan, Xobsv[9, :])

plt.xlabel("Time (s)")
plt.ylabel("MRP")
plt.title("flex eta state vs observer estimate")
plt.legend([
    "eta1 (true)", "eta1 (estimated)",
    "eta2 (true)", "eta2 (estimated)",
    "eta3 (true)", "eta3 (estimated)",
    "eta4 (true)", "eta4 (estimated)"
])
plt.show()

plt.figure()
plt.plot(tspan, X[6, :] - Xobs[6, :])
plt.plot(tspan, X[7, :] - Xobs[7, :])
plt.plot(tspan, X[8, :] - Xobs[8, :])
plt.plot(tspan, X[9, :] - Xobs[9, :])

plt.xlabel("Time (s)")
plt.ylabel("flex eta estimation error (rad/s)")
plt.title("flex eta estimation error (angular velocity)")
plt.legend([
    "eta1 error", "eta2 error", "eta3 error"
])
plt.show()
